# Projeter

| Module | Durée | Prérequis |
|---|---|---|
| M6 – Orthogonalité et moindres carrés | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

**Première situation.** Léa randonne hors sentier, dans le brouillard. Son téléphone indique sa position, mais pas de réseau. Elle sait que le sentier balisé part du refuge en ligne droite vers le col. Elle veut le rejoindre **par le plus court chemin**.

**Seconde situation.** Une lampe d'ambiance ne contient que deux sortes de diodes : des diodes **ambre** et des diodes **bleutées**. On règle l'intensité de chacune. Un client voudrait que la lampe produise un « blanc chaud » précis. C'est impossible exactement : quelle est la couleur la **plus proche** que la lampe peut produire ?

**La question commune : comment trouver, parmi tous les points d'une droite ou d'un plan, celui qui est le plus proche d'un point donné ?**

**Activité** : Pourquoi la seconde situation est-elle, mathématiquement, du même type que la première ? Dans quel espace vivent les couleurs ? Que représente le « plan » ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Dans le repère de la carte, en mètres, le refuge est à l'origine, le col au point $(800\,;\,450)$, et Léa au point $(520\,;\,610)$.

In [ ]:
pari_distance_au_sentier = 300     # en mètres (modifiez)

## ③ Abstraire : la projection orthogonale

### Sur une droite

Le point le plus proche est le **pied de la perpendiculaire** : le segment qui le relie au point de départ forme un angle droit avec la droite. Ce point s'appelle la **projection orthogonale**.

![Schéma : projection p du vecteur v sur la droite engendrée par u ; le reste r = v - p est perpendiculaire à u](../../assets/schemas/schema-projection-droite.svg)

Si la droite est engendrée par $\mathbf{u}$, la projection s'écrit $\mathbf{p} = t\,\mathbf{u}$. La condition d'angle droit $\mathbf{u} \cdot (\mathbf{v} - t\,\mathbf{u}) = 0$ donne directement

$$
t = \frac{\mathbf{u} \cdot \mathbf{v}}{\mathbf{u} \cdot \mathbf{u}}, \qquad \mathbf{p} = \frac{\mathbf{u} \cdot \mathbf{v}}{\mathbf{u} \cdot \mathbf{u}}\,\mathbf{u}.
$$

Pourquoi est-ce le plus proche ? Pour tout autre point $\mathbf{q}$ de la droite, le triangle formé par $\mathbf{v}$, $\mathbf{p}$ et $\mathbf{q}$ est rectangle en $\mathbf{p}$. Par le théorème de Pythagore, $\lVert \mathbf{v} - \mathbf{q} \rVert^2 = \lVert \mathbf{v} - \mathbf{p} \rVert^2 + \lVert \mathbf{p} - \mathbf{q} \rVert^2 \geq \lVert \mathbf{v} - \mathbf{p} \rVert^2$.

**Activité** : Projetez $\mathbf{v} = (3\,;\,4)$ sur la droite engendrée par $\mathbf{u} = (2\,;\,1)$. Vérifiez que le reste $\mathbf{r} = \mathbf{v} - \mathbf{p}$ est orthogonal à $\mathbf{u}$.

Indiquez vos réponses et justifiez-les.

### Sur un plan

Si le plan est engendré par deux vecteurs $\mathbf{a}_1$ et $\mathbf{a}_2$, ses points s'écrivent $x_1\mathbf{a}_1 + x_2\mathbf{a}_2 = A\mathbf{x}$, où $A$ a pour colonnes $\mathbf{a}_1$ et $\mathbf{a}_2$. Le reste doit être orthogonal aux **deux** vecteurs, ce qui s'écrit en une seule ligne :

$$
A^{\mathsf{T}}(\mathbf{b} - A\hat{\mathbf{x}}) = \mathbf{0}
\qquad\Longleftrightarrow\qquad
A^{\mathsf{T}}A\,\hat{\mathbf{x}} = A^{\mathsf{T}}\mathbf{b}.
$$

![Schéma : le vecteur b sort du plan des colonnes de A ; sa projection A x chapeau est le point du plan le plus proche, et le reste est orthogonal au plan](../../assets/schemas/schema-projection-plan.svg)

Ce sont les **équations normales** (« normal » veut dire ici « perpendiculaire »). Pour une droite, $A$ n'a qu'une colonne $\mathbf{u}$ et l'on retrouve la formule précédente.

## ④ Calculer

### Rejoindre le sentier

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def projeter_sur_droite(v, u):
    t = (u @ v) / (u @ u)
    return t, t * u

col = np.array([800., 450.])
lea = np.array([520., 610.])
t, p = projeter_sur_droite(lea, col)
r = lea - p
print(f"Point du sentier le plus proche : ({p[0]:.0f} ; {p[1]:.0f}), soit {t:.0%} du chemin vers le col")
print(f"Distance à parcourir            : {np.linalg.norm(r):.0f} m")
print(f"Vérification de l'angle droit   : col · reste = {col @ r:.2e}")

### Corroborer par une autre méthode

On vérifie sans formule : on essaie 10 001 points régulièrement espacés sur le sentier et on garde le plus proche.

In [ ]:
ts = np.linspace(0, 1, 10_001)
points = np.outer(ts, col)                      # une ligne par point essayé
distances = np.linalg.norm(points - lea, axis=1)
k = np.argmin(distances)
print(f"Recherche exhaustive : point ({points[k, 0]:.0f} ; {points[k, 1]:.0f}), distance {distances[k]:.1f} m")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
ax = axes[0]
ax.plot([0, col[0]], [0, col[1]], color=LIOUAI["gris"], lw=3, label="sentier")
ax.plot(*lea, "o", color=LIOUAI["navy"], ms=8, label="Léa")
ax.plot([lea[0], p[0]], [lea[1], p[1]], "--", color=LIOUAI["jaune"], lw=2, label="plus court chemin")
ax.plot(*p, "s", color=LIOUAI["vert"], ms=8, label="projection")
ax.annotate("refuge", (0, 0), textcoords="offset points", xytext=(5, -14))
ax.annotate("col", col, textcoords="offset points", xytext=(5, 5))
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(loc="upper left"); ax.set_title("La carte (en mètres)")
ax = axes[1]
ax.plot(ts * 100, distances, color=LIOUAI["navy"])
ax.axvline(t * 100, color=LIOUAI["vert"], ls="--")
ax.set_xlabel("position sur le sentier (% du chemin vers le col)"); ax.set_ylabel("distance à Léa (m)")
ax.set_title("La distance est minimale à la projection"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### La lampe à deux diodes

Les couleurs sont des vecteurs $(R\,;\,G\,;\,B)$, comme au module M1. Une diode ambre réglée à l'intensité $x_1$ et une diode bleutée à l'intensité $x_2$ produisent la couleur $x_1\mathbf{a}_1 + x_2\mathbf{a}_2$ : les lumières s'additionnent. Les couleurs possibles forment donc le **plan** engendré par $\mathbf{a}_1$ et $\mathbf{a}_2$.

In [ ]:
ambre = np.array([1.00, 0.55, 0.10])
bleutee = np.array([0.55, 0.75, 1.00])
blanc_chaud = np.array([1.00, 0.85, 0.65])     # la couleur demandée

A = np.column_stack([ambre, bleutee])
x_chapeau = np.linalg.solve(A.T @ A, A.T @ blanc_chaud)      # équations normales
couleur_obtenue = A @ x_chapeau
reste = blanc_chaud - couleur_obtenue

print("Intensités des diodes  :", x_chapeau.round(3))
print("Couleur obtenue        :", couleur_obtenue.round(3))
print("Écart avec la demande  :", reste.round(3), f" (longueur {np.linalg.norm(reste):.3f})")
print("Reste orthogonal aux deux diodes ? A^T r =", (A.T @ reste).round(12) + 0.0)

fig, ax = plt.subplots(figsize=(8, 1.8))
for k, (c, nom) in enumerate([(ambre, "diode ambre"), (bleutee, "diode bleutée"),
                              (blanc_chaud, "demandé"), (couleur_obtenue, "le plus proche possible")]):
    ax.add_patch(plt.Rectangle((k * 1.1, 0), 1, 1, color=np.clip(c, 0, 1)))
    ax.text(k * 1.1 + 0.5, -0.18, nom, ha="center", va="top", fontsize=9)
ax.set_xlim(-0.1, 4.4); ax.set_ylim(-0.5, 1.05); ax.axis("off")
plt.tight_layout(); plt.show()

### La matrice de projection

On peut ranger toute l'opération dans une seule matrice $P = A(A^{\mathsf{T}}A)^{-1}A^{\mathsf{T}}$ : pour tout vecteur $\mathbf{b}$, $P\mathbf{b}$ est sa projection sur le plan. Deux propriétés la caractérisent.

In [ ]:
P = A @ np.linalg.inv(A.T @ A) @ A.T
print("P b = projection ?           ", np.allclose(P @ blanc_chaud, couleur_obtenue))
print("Projeter deux fois = une fois (P P = P) ?", np.allclose(P @ P, P))
print("P est symétrique (P^T = P) ? ", np.allclose(P.T, P))
print("Une couleur déjà dans le plan ne bouge pas ?", np.allclose(P @ (0.3 * ambre + 0.6 * bleutee), 0.3 * ambre + 0.6 * bleutee))

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_distance_au_sentier} m ; distance réelle : {np.linalg.norm(lea - p):.0f} m")

**Activité** :

1. Indiquez à Léa, en une phrase, la direction à prendre et la distance à parcourir. Que se passerait-il si la projection tombait **au-delà** du col, avec $t > 1$ ?
2. Dites au client ce que la lampe peut faire pour son blanc chaud, en langage courant. La différence se verra-t-elle ? Sur quelle composante est-elle la plus grande ?
3. Pourquoi « projeter deux fois revient à projeter une fois » ? Donnez une explication géométrique, sans calcul.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèses.** Le sentier est-il vraiment rectiligne ? La distance « à vol d'oiseau » est-elle la bonne mesure en montagne ? Pour la lampe, la distance entre deux couleurs dans l'espace $(R\,;\,G\,;\,B)$ correspond-elle à la différence perçue par l'œil ?
- **Une contrainte oubliée.** Une intensité de diode ne peut pas être négative. Trouvez une couleur demandée pour laquelle la projection donne une intensité négative. Que faudrait-il faire alors ?
- **Généraliser.** Écrivez une fonction `projeter(A, b)` qui renvoie les coefficients $\hat{\mathbf{x}}$, la projection et le reste, pour une matrice $A$ à un nombre quelconque de colonnes. Testez-la sur le sentier (une colonne) et sur une lampe à trois diodes rouge, verte, bleue : quel est alors l'écart ? Pourquoi ?

In [ ]:
def projeter(A, b):
    # À compléter : renvoie (x_chapeau, projection, reste)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def projeter(A, b):
    A = np.asarray(A, dtype=float).reshape(len(b), -1)       # une seule colonne acceptée aussi
    x = np.linalg.solve(A.T @ A, A.T @ b)
    projection = A @ x
    return x, projection, b - projection

x, proj, r = projeter(col, lea)
print("Sentier : t =", x.round(4), " distance =", np.linalg.norm(r).round(1), "m")

rvb = np.eye(3)
x, proj, r = projeter(rvb, blanc_chaud)
print("Lampe RVB : intensités", x, " écart", np.linalg.norm(r))
print("  -> trois couleurs indépendantes engendrent tout l'espace : l'écart est nul.")

bleu_vif = np.array([0.1, 0.2, 0.9])
x, proj, r = projeter(A, bleu_vif)
print("Bleu vif demandé : intensités", x.round(3), " -> une intensité négative, impossible en pratique.")
print("  On peut alors chercher la meilleure solution avec intensités positives (moindres carrés sous contrainte).")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Décrire le concept": 1,
    "Expliquer visuellement": 1,
    "Identifier les concepts pertinents et leurs relations": 1,
    "Corroborer ses résultats": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "6.1-projeter", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")